In [26]:
import os
test_dir = '/content/drive/MyDrive/lmu_adl_eval_history'
os.makedirs(test_dir, exist_ok=True)
test_file = os.path.join(test_dir, '_write_test.txt')
with open(test_file, 'w') as f:
    f.write('ok')
print(f'Write successful: {os.path.exists(test_file)}')
# os.remove(test_file)


Write successful: True


# SGHMC Evaluation: Pretrained-Prior vs Zero-Centered-Prior

Runs both **automated metrics** (BLEU / ROUGE / Perplexity) and **LLM-as-judge**
(Quality / Diversity / Relevance) on SGHMC models with two prior configurations.

## 1. Setup

In [ ]:
# Clone repo (kernel runs on Colab) and mount Drive
import os
if not os.path.isdir('/content/adl-bnn-textgen'):
    !git clone https://github.com/ssophiee/adl-bnn-textgen /content/adl-bnn-textgen
else:
    !git -C /content/adl-bnn-textgen pull --ff-only
    print('Repo updated.')

from google.colab import drive
drive.mount('/content/drive')

In [17]:
import os

# Resolve project root by looking for a known marker file
def find_project_root():
    # Check common locations
    candidates = [
        os.getcwd(),
        os.path.abspath(os.path.join(os.getcwd(), '..')),
        '/content/adl-bnn-textgen',
    ]
    for c in candidates:
        if os.path.isfile(os.path.join(c, 'config.py')):
            return c
    raise FileNotFoundError('Could not find project root (looked for config.py)')

PROJECT_ROOT = find_project_root()
print(f'Project root: {PROJECT_ROOT}')
!git -C "{PROJECT_ROOT}" log --oneline -3

Project root: /content/adl-bnn-textgen
28175f6 (HEAD -> main, origin/main, origin/HEAD) fix(prior): use magnitude-scaled prior std and sync global CONFIG
a76e82a docs: add explicit prior configuration section to README and report
d69cca3 fix(pipeline): remove parallel llm evaluation


In [ ]:
# Write .env BEFORE any project imports (config.py reads it at import time)
env_content = f"""BASE_DIR={PROJECT_ROOT}
MODEL_PATH=/content/drive/MyDrive/baseline/baseline_model_2k.pt
META_PATH=/content/drive/MyDrive/baseline/nanogpt_meta.pkl
DATA_DIR={PROJECT_ROOT}/data/tokenized
BNN_MODEL_PATH=/content/drive/MyDrive/baseline/baseline_model_2k.pt
DEVICE=cuda
WANDB_AVAILABLE=false
"""

env_path = os.path.join(PROJECT_ROOT, '.env')
with open(env_path, 'w') as f:
    f.write(env_content)

print(env_content)
print(f'Environment file written to: {env_path}')

In [19]:
!pip install -q python-dotenv posteriors evaluate rouge_score datasets

## 2. Discover models

In [20]:
import glob, os

SGHMC_BASE = '/content/drive/MyDrive/samplers/sghmc_sampler'

pretrained_models = sorted(glob.glob(f'{SGHMC_BASE}/pretrained-prior/run_*/sghmc_model.pt'))
zero_models = sorted(glob.glob(f'{SGHMC_BASE}/zero-centered-prior/run_*/sghmc_model.pt'))

print(f'Pretrained-prior models ({len(pretrained_models)}):')
for p in pretrained_models:
    print(f'  {p}')

print(f'\nZero-centered-prior models ({len(zero_models)}):')
for p in zero_models:
    print(f'  {p}')

all_models = pretrained_models + zero_models
assert len(all_models) == 4, f'Expected 4 models, found {len(all_models)}. Check the paths above.'

Pretrained-prior models (2):
  /content/drive/MyDrive/samplers/sghmc_sampler/pretrained-prior/run_20260224-230657/sghmc_model.pt
  /content/drive/MyDrive/samplers/sghmc_sampler/pretrained-prior/run_20260224-232547/sghmc_model.pt

Zero-centered-prior models (2):
  /content/drive/MyDrive/samplers/sghmc_sampler/zero-centered-prior/run_20260224-231606/sghmc_model.pt
  /content/drive/MyDrive/samplers/sghmc_sampler/zero-centered-prior/run_20260224-233423/sghmc_model.pt


## 3. LLM-as-Judge Evaluation (Generation + Scoring)

In [ ]:
# QUICK TEST: 1 model, 1 prompt, single param combo (change_params=False)
# Results saved to Drive so they survive runtime disconnects
model_types = ['bayesian']

DRIVE_OUTPUT = '/content/drive/MyDrive/lmu_adl_eval_history'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

aggregated, full_results = run_evaluation_pipeline(
    test_prompts=test_prompts[:1],       # single prompt
    model_paths=all_models[:1],          # first model only
    model_types=model_types,
    change_params=False,                 # no param sweep
    output_path=f'{DRIVE_OUTPUT}/generation_results.json',
    use_local_qwen=True,
    device='cuda',
    use_external_data=True,
)

In [24]:
# Print summary
from pathlib import Path

print('\n' + '='*80)
print('LLM-Judge Results Summary')
print('='*80)
for model_path, scores in aggregated.items():
    run_dir = Path(model_path).parent.name
    prior_type = Path(model_path).parent.parent.name
    print(f'\n{prior_type} / {run_dir}:')
    print(f'  Quality:   {scores["overall_avg_quality"]:.2f}')
    print(f'  Diversity: {scores["overall_avg_diversity"]:.2f}')
    print(f'  Relevance: {scores["overall_avg_relevance"]:.2f}')
    print(f'  Generations: {scores["total_generations"]}')


LLM-Judge Results Summary

pretrained-prior / run_20260224-230657:
  Quality:   5.50
  Diversity: 6.00
  Relevance: 7.50
  Generations: 1


## 4. Automated Metrics (BLEU / ROUGE / Perplexity)

In [ ]:
import torch
from pathlib import Path

from src.nanogpt_utils import load_model, load_tokenizer
from src.generation_utils import load_checkpoint_for_generation
from scripts.bayesian_evaluator import BayesianNanoGPTEvaluator, evaluate_bayesian_splits

META_PATH = '/content/drive/MyDrive/baseline/nanogpt_meta.pkl'
DATA_DIR  = os.path.join(PROJECT_ROOT, 'data/tokenized')
DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'

stoi, itos = load_tokenizer(Path(META_PATH))
print(f'Vocab size: {len(itos)}, Device: {DEVICE}')

In [ ]:
import json, traceback, time

AUTO_METRICS_PATH = f'{DRIVE_OUTPUT}/automated_metrics.json'

def _save_auto_checkpoint():
    with open(AUTO_METRICS_PATH, 'w') as f:
        json.dump(auto_results, f, indent=2, default=str)

# Auto-resume: load previous results from Drive
auto_results = {}
if os.path.exists(AUTO_METRICS_PATH):
    with open(AUTO_METRICS_PATH, 'r') as f:
        auto_results = json.load(f)
    print(f'Resumed {len(auto_results)} previous model results from Drive')

eval_config = {
    'data_dir': DATA_DIR,
    'splits': ['train', 'val'],
    'batch_size': 16,
    'max_eval_samples': 500,
    'num_text_samples': 30,
    'prompt_length': 20,
    'generation_length': 30,
    'max_tokens': None,
}

METRIC_STEPS = ['perplexity_external_gpt2', 'bleu']  # bleu step also does rouge

for model_path in all_models:
    run_dir = Path(model_path).parent.name
    prior_type = Path(model_path).parent.parent.name
    label = f'{prior_type}/{run_dir}'

    # Check which (split, metric) pairs are still needed
    existing = auto_results.get(label, {})
    work = []
    for split in eval_config['splits']:
        split_data = existing.get(split, {})
        if isinstance(split_data, dict) and 'error' not in split_data:
            for step in METRIC_STEPS:
                if step not in split_data:
                    work.append((split, step))
        else:
            work.extend([(split, step) for step in METRIC_STEPS])

    if not work:
        print(f'\nSKIP (cached): {label}')
        continue

    print(f'\n{"="*60}')
    print(f'Evaluating: {label} ({len(work)} steps remaining)')
    print(f'{"="*60}')

    model, _ = load_model(Path('/content/drive/MyDrive/baseline/baseline_model_2k.pt'), DEVICE)
    ckpt = load_checkpoint_for_generation(model_path, device=DEVICE)
    collected_samples = ckpt.get('collected_samples', [])
    print(f'  Collected samples: {len(collected_samples)}')

    evaluator = BayesianNanoGPTEvaluator(
        model=model, stoi=stoi, itos=itos,
        sampler_type='sghmc',
        collected_samples=collected_samples,
        device=DEVICE,
        num_posterior_samples=min(10, len(collected_samples)),
    )

    if label not in auto_results:
        auto_results[label] = {'model_path': model_path, 'prior_type': prior_type}

    for split in eval_config['splits']:
        if split not in auto_results[label] or not isinstance(auto_results[label].get(split), dict):
            auto_results[label][split] = {'split': split, 'sampler_type': 'sghmc'}

        split_data = auto_results[label][split]
        if 'error' in split_data:
            split_data = {'split': split, 'sampler_type': 'sghmc'}
            auto_results[label][split] = split_data

        data = evaluator.load_data(eval_config['data_dir'], split, eval_config.get('max_tokens'))
        max_ppl_batches = min(100, eval_config['max_eval_samples'] // eval_config['batch_size'])

        # Step 1: External GPT-2 perplexity
        if 'perplexity_external_gpt2' not in split_data:
            print(f'  [{split}] perplexity_gpt2...', end=' ', flush=True)
            t0 = time.time()
            try:
                ppl_ext = evaluator.calculate_perplexity_external_gpt2(data, eval_config['batch_size'], max_batches=max_ppl_batches)
                split_data['perplexity_external_gpt2'] = ppl_ext if ppl_ext is not None else 0.0
                print(f'{split_data["perplexity_external_gpt2"]:.1f} ({time.time()-t0:.0f}s)')
            except Exception as e:
                split_data['perplexity_external_gpt2'] = 0.0
                print(f'ERROR: {e}')
            _save_auto_checkpoint()

        # Step 2: Text generation + BLEU + ROUGE
        if 'bleu' not in split_data:
            print(f'  [{split}] generate + bleu/rouge...', end=' ', flush=True)
            t0 = time.time()
            try:
                refs, preds = evaluator.generate_samples_for_metrics(
                    data, eval_config['num_text_samples'],
                    eval_config['prompt_length'], eval_config['generation_length'])
                if refs and preds:
                    bleu = evaluator.calculate_bleu_score(refs, preds)
                    split_data['bleu'] = bleu if bleu is not None else 0.0
                    rouge = evaluator.calculate_rouge_score(refs, preds)
                    split_data.update(rouge)
                else:
                    split_data.update({'bleu': 0.0, 'rouge1': 0.0, 'rouge2': 0.0, 'rougeL': 0.0})
                print(f'BLEU={split_data["bleu"]:.4f} ROUGE-2={split_data.get("rouge2",0):.4f} ({time.time()-t0:.0f}s)')
            except Exception as e:
                split_data.update({'bleu': 0.0, 'rouge1': 0.0, 'rouge2': 0.0, 'rougeL': 0.0})
                print(f'ERROR: {e}')
            _save_auto_checkpoint()

    print(f'  Done: {label}')

print(f'\nAll {len(auto_results)} models evaluated.')

## 5. Save Results

In [ ]:
from datetime import datetime

print(f'Automated metrics already saved to: {AUTO_METRICS_PATH}')
print(f'LLM-judge results saved to: {DRIVE_OUTPUT}/generation_results*.json')
print(f'\nAll results on Drive at {datetime.now():%Y-%m-%d %H:%M:%S}')

## 6. Quick Comparison Table

In [ ]:
import pandas as pd

rows = []
for label, data in auto_results.items():
    for split in ['train', 'val']:
        m = data.get(split, {})
        if isinstance(m, dict) and 'error' not in m:
            rows.append({
                'model': label,
                'split': split,
                'bleu': m.get('bleu', 0),
                'rouge1': m.get('rouge1', 0),
                'rouge2': m.get('rouge2', 0),
                'rougeL': m.get('rougeL', 0),
                'ppl_gpt2': m.get('perplexity_external_gpt2', 0),
            })

df = pd.DataFrame(rows)
if not df.empty:
    display(df.round(4))
else:
    print('No results to display.')